# Coral Reef Water Quality — Dataset Generation (2004–2024)

Exports **four datasets** from Google Earth Engine to Google Drive.

> **Note:** Start year is 2004 — MODIS Aqua (the source for `chlor_a`, `Rrs_443`, `Rrs_490`) launched May 2002 with stable data only from 2003+. Using 2004 ensures complete ocean color coverage across all bands.

| # | Dataset | Scope | Resolution | Key Columns |
|---|---------|-------|-----------|-------------|
| 1 | Andaman 20-Year | 4 ANI reef sites | Weekly | chlor_a, total_nobs, SST, precip |
| 2 | Andaman Harmonized | 4 ANI reef sites | Weekly | + Rrs_443/490, u_wind, v_wind, mslp |
| 3 | Global 20-Year | 6 global reef sites | Monthly | chlor_a, total_nobs, SST, precip |
| 4 | Global Harmonized | 6 global reef sites | Monthly | + Rrs_443/490, u_wind, v_wind, mslp |


**Run cells in order (2 → 3 → 4 → 5 → 6 → 7). Then re-run cell 8 to monitor export progress.**All CSVs land in `Google Drive → EarthEngine_Data/`.

In [2]:
# ── Cell 2: Mount Drive + Authenticate + Initialise GEE ──────────────────────
from google.colab import drive
import ee

drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='coral-relief-using-ml')
print("Google Earth Engine initialised successfully.")

Mounted at /content/drive
Google Earth Engine initialised successfully.


In [3]:
# ── Cell 3: Common Parameters ─────────────────────────────────────────────────

# Date range: 2004-2024
# (MODIS Aqua chlor_a / Rrs bands only reliably available from 2002+; 2004 is safe)
START_DATE = ee.Date('2004-01-01')
END_DATE   = ee.Date('2024-12-31')

DRIVE_FOLDER = 'EarthEngine_Data'

# ── Andaman reef sites (4 locations, 5 km buffer) ─────────────────────────────
andaman_pts = [
    {'coords': [92.74, 11.66], 'label': 'Port_Blair'},
    {'coords': [93.00, 11.98], 'label': 'Havelock'},
    {'coords': [93.03, 11.90], 'label': 'Neil'},
    {'coords': [92.65, 11.55], 'label': 'Wandoor'},
]
andaman_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(r['coords']).buffer(5000), {'label': r['label']})
    for r in andaman_pts
])

# ── Global reef sites (6 locations, 10 km buffer) ─────────────────────────────
global_pts = [
    {'coords': [-81.4,   24.6],  'label': 'Florida_USA'},
    {'coords': [147.7,  -18.2],  'label': 'GBR_Australia'},
    {'coords': [38.5,    20.3],  'label': 'RedSea_MiddleEast'},
    {'coords': [55.5,    -4.6],  'label': 'Seychelles_Africa'},
    {'coords': [24.0,    35.0],  'label': 'Mediterranean_Europe'},
    {'coords': [-157.8,  21.3],  'label': 'Hawaii_USA'},
]
global_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(r['coords']).buffer(10000), {'label': r['label']})
    for r in global_pts
])

# ── GEE Collections ───────────────────────────────────────────────────────────
OC_BASIC = (ee.ImageCollection('COPERNICUS/MARINE/SATELLITE_OCEAN_COLOR/V6')
              .select(['chlor_a', 'total_nobs']))

OC_FULL  = (ee.ImageCollection('COPERNICUS/MARINE/SATELLITE_OCEAN_COLOR/V6')
              .select(['chlor_a', 'Rrs_443', 'Rrs_490', 'total_nobs']))

ERA5_BASIC = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
               .select(['sea_surface_temperature', 'total_precipitation']))

ERA5_FULL  = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
               .select([
                   'sea_surface_temperature',
                   'total_precipitation',
                   'u_component_of_wind_10m',
                   'v_component_of_wind_10m',
                   'mean_sea_level_pressure',
               ]))

# ── Week / month counts ───────────────────────────────────────────────────────
N_WEEKS  = END_DATE.difference(START_DATE, 'week').toInt()
N_MONTHS = END_DATE.difference(START_DATE, 'month').toInt()

print(f"Date range : {START_DATE.format('YYYY-MM-dd').getInfo()}  →  {END_DATE.format('YYYY-MM-dd').getInfo()}")
print(f"Weeks      : {N_WEEKS.getInfo()}")
print(f"Months     : {N_MONTHS.getInfo()}")
print(f"Andaman sites : {len(andaman_pts)}  |  Global sites : {len(global_pts)}")

Date range : 2004-01-01  →  2024-12-31
Weeks      : 1095
Months     : 251
Andaman sites : 4  |  Global sites : 6


In [4]:
# ── Cell 4: [1/4] Andaman 20-Year Dataset (2004–2024) ─────────────────────────
#   Weekly composites, 4 reef sites, basic columns
#   Output columns: chlor_a, total_nobs, sea_surface_temperature,
#                   total_precipitation, label, week_start

def make_andaman_20y(n):
    wk_start = START_DATE.advance(ee.Number(n), 'week')
    wk_end   = wk_start.advance(7, 'day')
    oc_w   = OC_BASIC.filterDate(wk_start, wk_end).mean()
    era5_w = ERA5_BASIC.filterDate(wk_start, wk_end).mean()
    return oc_w.addBands(era5_w).set('week_start', wk_start.format('YYYY-MM-dd'))

def sample_andaman(img):
    return (img.reduceRegions(
                collection=andaman_fc,
                reducer=ee.Reducer.mean(),
                scale=4000)
            .map(lambda f: f.set('week_start', img.get('week_start'))))

weekly_20y = ee.ImageCollection(
    ee.List.sequence(0, N_WEEKS.subtract(1)).map(make_andaman_20y)
)

task_and_20y = ee.batch.Export.table.toDrive(
    collection=weekly_20y.map(sample_andaman).flatten(),
    description='Andaman_WQ_20Year_Dataset',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Andaman_WQ_20Year_Dataset',
    fileFormat='CSV',
)
task_and_20y.start()
print("[1/4] Andaman 20-Year export started.")
print("      Task ID:", task_and_20y.id)

[1/4] Andaman 20-Year export started.
      Task ID: YARAZX7YP5ESY7XTHUYQBMSD


In [5]:
# ── Cell 5: [2/4] Andaman Harmonized Dataset (2004–2024) ──────────────────────
#   Weekly composites, 4 reef sites, full columns
#   Output columns: chlor_a, Rrs_443, Rrs_490, total_nobs,
#                   sst, precip, u_wind, v_wind, mslp, label, week_start

def make_andaman_harm(n):
    wk_start = START_DATE.advance(ee.Number(n), 'week')
    wk_end   = wk_start.advance(7, 'day')
    oc_w   = OC_FULL.filterDate(wk_start, wk_end).mean()
    era5_w = (ERA5_FULL.filterDate(wk_start, wk_end).mean()
                       .rename(['sst', 'precip', 'u_wind', 'v_wind', 'mslp']))
    return oc_w.addBands(era5_w).set('week_start', wk_start.format('YYYY-MM-dd'))

weekly_harm = ee.ImageCollection(
    ee.List.sequence(0, N_WEEKS.subtract(1)).map(make_andaman_harm)
)

task_and_harm = ee.batch.Export.table.toDrive(
    collection=weekly_harm.map(sample_andaman).flatten(),
    description='Andaman_WQ_Harmonized_20Year',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Andaman_WQ_Harmonized_20Year',
    fileFormat='CSV',
)
task_and_harm.start()
print("[2/4] Andaman Harmonized export started.")
print("      Task ID:", task_and_harm.id)

[2/4] Andaman Harmonized export started.
      Task ID: 5FNBDQB26MWNG5SFOVXDEOQG


In [6]:
# ── Cell 6: [3/4] Global 20-Year Dataset (2004–2024) ──────────────────────────
#   Monthly composites, 6 global reef sites, basic columns
#   Output columns: chlor_a, total_nobs, sea_surface_temperature,
#                   total_precipitation, label, week_start

def make_global_20y(n):
    mo_start = START_DATE.advance(ee.Number(n), 'month')
    mo_end   = mo_start.advance(1, 'month')
    oc_m   = OC_BASIC.filterDate(mo_start, mo_end).mean()
    era5_m = ERA5_BASIC.filterDate(mo_start, mo_end).mean()
    return oc_m.addBands(era5_m).set('week_start', mo_start.format('YYYY-MM-dd'))

def sample_global(img):
    return (img.reduceRegions(
                collection=global_fc,
                reducer=ee.Reducer.mean(),
                scale=4000)
            .map(lambda f: f.set('week_start', img.get('week_start'))))

monthly_20y = ee.ImageCollection(
    ee.List.sequence(0, N_MONTHS.subtract(1)).map(make_global_20y)
)

task_glob_20y = ee.batch.Export.table.toDrive(
    collection=monthly_20y.map(sample_global).flatten(),
    description='Global_WQ_20Year_Dataset',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Global_WQ_20Year_Dataset',
    fileFormat='CSV',
)
task_glob_20y.start()
print("[3/4] Global 20-Year export started.")
print("      Task ID:", task_glob_20y.id)

[3/4] Global 20-Year export started.
      Task ID: M7SPYPVWKMZUUSKXTAAKH2DD


In [7]:
# ── Cell 7: [4/4] Global Harmonized Dataset (2004–2024) ───────────────────────
#   Monthly composites, 6 global reef sites, full columns
#   Output columns: chlor_a, Rrs_443, Rrs_490, total_nobs,
#                   sst, precip, u_wind, v_wind, mslp, label, week_start

def make_global_harm(n):
    mo_start = START_DATE.advance(ee.Number(n), 'month')
    mo_end   = mo_start.advance(1, 'month')
    oc_m   = OC_FULL.filterDate(mo_start, mo_end).mean()
    era5_m = (ERA5_FULL.filterDate(mo_start, mo_end).mean()
                       .rename(['sst', 'precip', 'u_wind', 'v_wind', 'mslp']))
    return oc_m.addBands(era5_m).set('week_start', mo_start.format('YYYY-MM-dd'))

monthly_harm = ee.ImageCollection(
    ee.List.sequence(0, N_MONTHS.subtract(1)).map(make_global_harm)
)

task_glob_harm = ee.batch.Export.table.toDrive(
    collection=monthly_harm.map(sample_global).flatten(),
    description='Global_WQ_Harmonized_20Year',
    folder=DRIVE_FOLDER,
    fileNamePrefix='Global_WQ_Harmonized_20Year',
    fileFormat='CSV',
)
task_glob_harm.start()
print("[4/4] Global Harmonized export started.")
print("      Task ID:", task_glob_harm.id)

[4/4] Global Harmonized export started.
      Task ID: TRYMY4OAOVPZGLQEA6RFVYFQ


In [11]:
# ── Cell 8: Monitor all 4 export tasks ────────────────────────────────────────
# Re-run this cell at any time to refresh status.

tasks = {
    '1/4  Andaman 20-Year    ': task_and_20y,
    '2/4  Andaman Harmonized ': task_and_harm,
    '3/4  Global  20-Year    ': task_glob_20y,
    '4/4  Global  Harmonized ': task_glob_harm,
}

all_done = True
print("=" * 60)
for name, t in tasks.items():
    status = t.status()
    state  = status.get('state', 'UNKNOWN')
    err    = status.get('error_message', '')
    if state == 'COMPLETED':
        icon = '[DONE]'
    elif state == 'FAILED':
        icon = '[FAIL]'
    else:
        icon = '[WAIT]'
        all_done = False
    msg = f"  {icon}  [{name}]  {state}"
    if err:
        msg += f"  <- {err}"
    print(msg)

print("=" * 60)
if all_done:
    print("\nAll 4 exports finished!")
    print("Files are in: Google Drive -> EarthEngine_Data/")
    print("  * Andaman_WQ_20Year_Dataset.csv")
    print("  * Andaman_WQ_Harmonized_20Year.csv")
    print("  * Global_WQ_20Year_Dataset.csv")
    print("  * Global_WQ_Harmonized_20Year.csv")
else:
    print("\nSome tasks still running -- re-run this cell in ~2 minutes to refresh.")

  [DONE]  [1/4  Andaman 20-Year    ]  COMPLETED
  [DONE]  [2/4  Andaman Harmonized ]  COMPLETED
  [DONE]  [3/4  Global  20-Year    ]  COMPLETED
  [DONE]  [4/4  Global  Harmonized ]  COMPLETED

All 4 exports finished!
Files are in: Google Drive -> EarthEngine_Data/
  * Andaman_WQ_20Year_Dataset.csv
  * Andaman_WQ_Harmonized_20Year.csv
  * Global_WQ_20Year_Dataset.csv
  * Global_WQ_Harmonized_20Year.csv
